# Data Analysis · Week 7
## Nested selection and logical operators

**TIA502 · School of Business · Instructor David Escobar-Castillejos**

Last session was one condition at a time. This one is how they join up, and when nesting beats
combining.

By the end of this notebook you will be able to:

1. Use the three logical operators: `and` demands both, `or` settles for one, `not` inverts.
2. Read a truth table and predict the result of a compound condition without running it.
3. Ask about membership with `in` and `not in`, instead of chaining comparisons with `or`.
4. Tell `is` from `==`.
5. Decide when to nest, and recognise the nesting that was really an `and`.

### How to use this notebook

Run the cells in order. Four fail on purpose or give a deliberately surprising result, and carry a
comment saying so.

All of it lands on one rule, worth carrying from the start: **if both branches of the nesting do
the same thing, it was an `and` in disguise.**

---
# Block 1 · Combining conditions

Three operators, and any rule can be built from them however complicated it sounds said out loud.

| Operator | What it demands | Example |
|---|---|---|
| `and` | That both conditions are true | `conversion >= 0.03 and clicks > 1000` |
| `or` | That at least one is true | `channel == "Meta" or channel == "Google"` |
| `not` | Inverts the result of the condition | `not campaign_active` |

## The truth table, generated

| A | B | `A and B` | `A or B` |
|---|---|---|---|
| `True` | `True` | `True` | `True` |
| `True` | `False` | `False` | `True` |
| `False` | `True` | `False` | `True` |
| `False` | `False` | `False` | `False` |

Rather than taking my word for it, build it.

In [ ]:
print(f"{'A':<7}{'B':<7}{'A and B':<10}{'A or B':<10}{'not A':<7}")
print("-" * 41)

for a in [True, False]:
    for b in [True, False]:
        print(f"{str(a):<7}{str(b):<7}{str(a and b):<10}{str(a or b):<10}{str(not a):<7}")

Four rows and that is all of it. `and` is only true on the first; `or` is false only on the last.

## A rule with two conditions

In [ ]:
conversion = 0.0342
clicks = 5074
channel = "Instagram"

if conversion >= 0.03 and clicks > 1000:
    print("The campaign qualifies for more budget.")
else:
    print("The campaign stays as it is.")

With `and`, if a single one fails, the whole rule fails. There is no middle ground.

**Why the second condition.** A high conversion over a hundred clicks means nothing: it could be
chance. Volume filters the noise, which is why the policy asks for both.

Swap the `and` for an `or` and look at who it approves.

In [ ]:
CANDIDATES = [
    ("Instagram", 0.0342, 5074),
    ("LinkedIn", 0.0205, 640),
    ("Newsletter", 0.0810, 62),   # sky-high conversion, almost no volume
    ("Display", 0.0021, 88400),   # enormous volume, dreadful conversion
]

print(f"{'Channel':<12}{'Conv.':>8}{'Clicks':>9}   with and       with or")
print("-" * 58)

for channel, conv, clicks in CANDIDATES:
    with_and = "approves" if (conv >= 0.03 and clicks > 1000) else "no"
    with_or = "approves" if (conv >= 0.03 or clicks > 1000) else "no"
    print(f"{channel:<12}{conv:>8.2%}{clicks:>9,}   {with_and:<15}{with_or}")

With `or`, Display approves: eighty-eight thousand clicks and a conversion of 0.21 %. It is the
campaign burning the most money in the group and the rule rewards it.

That is the cost of changing one word.

## Short-circuit evaluation

If the first condition of an `and` is false, Python **does not even read the second**. It saves
work and, more importantly, prevents errors.

In [ ]:
def check(name):
    """Announces itself, so you can see when it runs."""
    print(f"  (evaluating {name})")
    return True


print("With the first one false:")
result = False and check("second")
print("Result:", result)

print()
print("With the first one true:")
result = True and check("second")
print("Result:", result)

In the first case `check` never ran. That is not a curiosity: it is what lets you write conditions
that would be an error the other way round.

In [ ]:
reported_clicks = 0

# The right order: check it is not zero first, divide afterwards.
if reported_clicks > 0 and 38500 / reported_clicks < 10:
    print("Cost per click acceptable")
else:
    print("Not enough clicks to evaluate")

In [ ]:
# FAILS ON PURPOSE. The same pair of conditions, in the wrong order.
try:
    if 38500 / reported_clicks < 10 and reported_clicks > 0:
        print("Cost per click acceptable")
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)

The same rule, two behaviours. With the guard first, the division never happens; the other way
round, it blows up before reaching the guard.

**When one condition protects the other, it goes first.** That is not style, it is what makes the
program run.

## `not`

It inverts. It gets used rarely, and when it does, the variable should already read as a
statement.

In [ ]:
campaign_active = False

print("not campaign_active:", not campaign_active)

if not campaign_active:
    print("The campaign is paused, there is nothing to evaluate.")

`if not campaign_active:` reads almost like English. Comparing with `if campaign_active == False:`
works the same and reads worse.

---
# Block 2 · Membership and identity

Four more operators, and two of them will save you writing the same comparison five times.

| Operator | Question | Example | Result |
|---|---|---|---|
| `in` | Is it inside? | `channel in ["Meta", "Google"]` | `False` |
| `not in` | Is it outside? | `channel not in ["TikTok"]` | `True` |
| `is` | Is it the same object? | `cost is None` | `True` |
| `is not` | Is it another object? | `cost is not None` | `False` |

## Five comparisons, or one

Chained with `or` it looks like this:

```python
if (channel == "Meta" or
    channel == "Google" or
    channel == "Instagram" or
    channel == "TikTok"):
    print("Digital channel")
```

And with `in`:

In [ ]:
DIGITAL = ["Meta", "Google", "Instagram", "TikTok"]

channel = "Instagram"

if channel in DIGITAL:
    print("Digital channel")
else:
    print("Another channel")

The second reads at a glance and, above all, **the list can change without touching the
condition**. Adding a channel is adding an element.

In [ ]:
DIGITAL.append("LinkedIn")

for channel in ["Instagram", "LinkedIn", "Radio", "Billboard"]:
    print(f"{channel:<12} {'digital' if channel in DIGITAL else 'traditional'}")

`in` also works on text, where it asks whether one string is contained in another.

In [ ]:
title = "Sales analyst"

print('"analyst" in title :', "analyst" in title)
print('"Analyst" in title :', "Analyst" in title, "<- capitals matter")
print('"analyst" in title.lower() :', "analyst" in title.lower())

That is the direct ancestor of `.str.contains("manager", case=False)` from week 15.2, applied to a
single value instead of a column.

## `is` is not `==`

The most confused pair. `==` asks whether they are worth the same; `is` asks whether they are
**the same object**.

In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]

print("a == b :", a == b, "<- worth the same")
print("a is b :", a is b, "<- and not the same object")

c = a
print("c is a :", c is a, "<- c is another name for the same object")

Two lists with the same contents are equal and are not the same. It is the difference between two
sheets holding the same data and two tabs pointing at the same file.

**For numbers and text, always use `==`.** `is` on values gives results that depend on Python's
internal details and cannot be predicted.

In [ ]:
# FAILS ON PURPOSE, in the worst way: it sometimes works.
x = 256
y = 256
print("256 is 256 :", x is y)

x = 1000
y = 1000
print("1000 is 1000 :", x is y, "<- same code, different result")

print()
print("With == it is always predictable:", 1000 == 1000)

The same code with a different number gives a different result, because Python keeps small
integers in a table and reuses them. None of that is anything you should lean on.

`is` has one correct use and this is it:

In [ ]:
cost_per_click = None

print("cost is None     :", cost_per_click is None)
print("cost is not None :", cost_per_click is not None)

cost_per_click = 7.59
print("Now measured, is not None:", cost_per_click is not None)

`is None` and `is not None` are the correct way to ask about a missing value, because `None` is a
single unique object across the whole program.

---
# Block 3 · Nesting a decision

A decision inside another. Sometimes it is right, and sometimes it is an `and` written in the
longest possible way.

## A nesting that earns its place

In [ ]:
campaign_active = False
conversion = 0.061

if campaign_active:
    if conversion >= 0.05:
        action = "Raise the budget"
    elif conversion >= 0.03:
        action = "Hold"
    else:
        action = "Pause and review"
else:
    action = "Reactivate before evaluating"

print(action)

Here the nesting earns something real, for three reasons.

**The first question decides whether you continue.** If the campaign is off, its conversion means
nothing: it is data from when it was on. Asking about it would be a business error.

**Each inner branch does something different.** Three outcomes, not two identical ones.

**The outer `else`** covers the whole off case without repeating the three categories.

The trace with a paused campaign at 6 % conversion:

| Step | Condition | Result | `action` |
|---|---|---|---|
| 1 | `campaign_active` | `False` | – |
| 2 | The whole inner block | Not evaluated | – |
| 3 | The outer `else` | Runs | `Reactivate before evaluating` |

The 6 % conversion is never looked at. The nesting protects it from a decision that would make no
sense to take.

## The nesting that was an `and`

Now the opposite case. Read it and look for what is redundant.

In [ ]:
def approve_nested(conversion, clicks):
    """Three levels of indentation for a single question."""
    if conversion >= 0.03:
        if clicks > 1000:
            return "approves"
        else:
            return "does not approve"
    else:
        return "does not approve"


def approve_combined(conversion, clicks):
    """The same thing, on one line."""
    return "approves" if (conversion >= 0.03 and clicks > 1000) else "does not approve"


for conv, clicks in [(0.0342, 5074), (0.081, 62), (0.0021, 88400), (0.02, 500)]:
    a = approve_nested(conv, clicks)
    b = approve_combined(conv, clicks)
    print(f"{conv:>7.2%}{clicks:>8,}   {a:<20}{b:<20}{'same' if a == b else 'DIFFERENT'}")

Identical in all four cases, and one takes nine lines and the other one.

**The test: if both inner branches do the same thing, it was an `and`.** In `approve_nested`, the
inner `else` and the outer `else` return exactly the same thing, and that is the signal.

It is the most profitable review you can give your own code. It turns four levels of indentation
into one readable line.

## The rule, stated in full

You nest when:

1. The second question **only makes sense** if the first one held.
2. Each branch does **something different**.

If both inner branches end up doing the same thing, or if the second question can always be asked,
then it was not a nesting: it was a single condition joined with `and`.

---
## Four traps in compound conditions

### Writing `and` when you meant `or`

Read it out loud. "Both" is `and`, "either of the two" is `or`. Half the errors get caught that
way.

### Comparing against two values at once

**Predict before you run.** What does this print, if the channel is Instagram?

- **A.** `No match`, because it is neither of the two.
- **B.** `Match`, because a non-empty string evaluates as true.
- **C.** An error, because a comparison is missing.
- **D.** `Match`, because Python compares against both.

In [ ]:
# FAILS ON PURPOSE, raising nothing. This condition does not do what it looks like.
channel = "Instagram"

if channel == "Meta" or "Google":
    print("Match")
else:
    print("No match")

The answer is **B**, and it is one of the language's nastiest traps.

Python reads that as `(channel == "Meta") or ("Google")`. The first part is false, so it evaluates
the second: bare `"Google"`, a non-empty string, which counts as true.

The condition is true **always**, for any channel.

In [ ]:
for channel in ["Instagram", "Meta", "Radio", "anything at all"]:
    result = channel == "Meta" or "Google"
    print(f"{channel:<18} -> {result!r}")

It does not even return `True`: it returns the string `"Google"`, which inside an `if` counts as
true.

The two correct forms:

In [ ]:
channel = "Instagram"

print("Repeating the variable:", channel == "Meta" or channel == "Google")
print("With in:               ", channel in ("Meta", "Google"))

### Using `is` to compare values

You saw it above. For numbers and text, always `==`.

### Nesting without needing to

Three levels of indentation are almost always two conditions joined with `and` and one redundant
branch.

## All together: a real policy

In [ ]:
DIGITAL = ["Meta", "Google", "Instagram", "TikTok", "LinkedIn"]

def budget_decision(channel, active, conversion, clicks):
    """The full policy, with all three logical operators and membership."""
    if not active:
        return "Reactivate before evaluating"

    if channel not in DIGITAL:
        return "Outside policy, review by hand"

    if conversion >= 0.05 and clicks > 1000:
        return "Raise the budget"
    elif conversion >= 0.03 or clicks > 50000:
        return "Hold"
    else:
        return "Pause and review"


CASES = [
    ("Instagram", True, 0.061, 5074),
    ("Instagram", False, 0.061, 5074),
    ("Radio", True, 0.061, 5074),
    ("Display", True, 0.0021, 88400),
    ("LinkedIn", True, 0.0205, 640),
]

for channel, active, conv, clicks in CASES:
    state = "active" if active else "paused"
    print(f"{channel:<11}{state:<9}{conv:>7.2%}{clicks:>8,}   {budget_decision(channel, active, conv, clicks)}")

Three guards at the top and one decision at the bottom. No indentation goes past two levels, and
the function reads top to bottom like a list of rules.

That pattern, pulling the special cases out first and leaving the main logic at the end, is what
avoids deep nesting nearly every time.

---
# Exercises

The solutions sit at the very bottom of the notebook.

## Logical

### Exercise 1 · The truth table of `not` and of the combination

Generate with a loop the truth table for `A and not B` and for `not (A or B)`. Four rows each.

Then answer in a comment: does either of them match `not A or not B`?

### Exercise 2 · Reading it out loud

For each of these three rules, write the Python condition and the English sentence describing it:

1. Approve if the customer has more than two years **and** their balance is under 10,000.
2. Alert if the order exceeds 100,000 **or** the customer is new.
3. Reject if it is **not** on the authorised supplier list.

### Exercise 3 · The guard that protects

Write a condition that computes the cost per click only if there are clicks, using short-circuit
evaluation. Test it with zero clicks and with real clicks.

Then write it the other way round and confirm it blows up.

## Membership

### Exercise 4 · From five `or` to one `in`

Write a condition with four `or` clauses checking whether a month is in the last quarter, then the
same with `in`. Test both with six different months and check they agree.

### Exercise 5 · Searching inside text

With this list of job titles, print the ones containing the word "manager" ignoring capitals, and
separately the ones containing "analyst".

```python
TITLES = ["Sales analyst", "Brand Manager", "people manager",
          "Financial Analyst", "Recruiter", "Operations Manager"]
```

### Exercise 6 · `is` against `==`

Create two dictionaries with the same contents and compare them with `==` and with `is`. Then
assign one to the other and compare again.

Explain in a comment in which case `is` would be the right question.

## Nesting

### Exercise 7 · Collapsing a nesting

This code has three levels of indentation and two branches doing the same thing. Rewrite it as a
single condition.

```python
def can_ship(weight, destination, paid):
    if paid:
        if weight <= 20:
            if destination != "international":
                return "ship"
            else:
                return "do not ship"
        else:
            return "do not ship"
    else:
        return "do not ship"
```

Check with eight combinations that both versions agree.

### Exercise 8 · A policy that needs two conditions

Write a rule from your field that depends on at least two values: approving credit by income and
tenure, or prioritising an order by amount and by customer. Use `and`, `or` and `in` at least once
each.

Two levels of indentation maximum. If you need three, collapse with `and`.

The test: read it out loud to a classmate. If they have to ask "and or or?", the condition is badly
written.

---
## Three ideas to take away

**`and` demands both, `or` settles for one.** Reading the condition out loud catches half the
errors before the program runs.

**`in` replaces a row of `or` clauses.** And it lets the list of valid values change without
touching a single line of the condition.

**If both branches do the same thing, it was an `and`.** The most profitable review you can give
your code, and it turns four levels of indentation into one.

Next session is repetition, and the first midterm.

---
# Solutions

### Exercise 1

```python
print(f"{'A':<7}{'B':<7}{'A and not B':<14}{'not (A or B)':<15}{'not A or not B':<15}")
for a in [True, False]:
    for b in [True, False]:
        print(f"{str(a):<7}{str(b):<7}{str(a and not b):<14}"
              f"{str(not (a or b)):<15}{str(not a or not b):<15}")

# not (A or B) does not match not A or not B. The one that matches
# not A or not B is not (A and B). It is one of De Morgan's laws: negating a
# combination turns the and into an or and negates each part.
```

That law is worth recognising even without naming it. It turns up every time somebody tries to
negate a compound condition and only does half the job.

### Exercise 2

```python
tenure_years = 3
balance = 8500
order_amount = 128000
new_customer = False
AUTHORISED = ["Insumos SA", "Papelera del Norte", "Log Express"]
supplier = "Some Other Supplier"

# 1. "More than two years AND balance under ten thousand"
print("Approve:", tenure_years > 2 and balance < 10000)

# 2. "Order over a hundred thousand OR new customer"
print("Alert:", order_amount > 100000 or new_customer)

# 3. "NOT on the authorised list"
print("Reject:", supplier not in AUTHORISED)
```

The third can be written `not (supplier in AUTHORISED)` and means the same. `not in` exists
precisely because it reads better.

### Exercise 3

```python
spend = 38500

for clicks in [0, 5074]:
    if clicks > 0 and spend / clicks < 10:
        print(f"{clicks:>6} clicks -> cost per click acceptable")
    else:
        print(f"{clicks:>6} clicks -> not evaluable or cost too high")

# The other way round it blows up:
try:
    clicks = 0
    if spend / clicks < 10 and clicks > 0:
        print("never gets here")
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)
```

The correct version needs no extra `if` and no `try`. The guard inside the same `and` does all the
work, and that only functions because of short-circuit evaluation.

### Exercise 4

```python
LAST_QUARTER = ["oct", "nov", "dec"]

for month in ["jan", "jun", "sep", "oct", "nov", "dec"]:
    with_or = month == "oct" or month == "nov" or month == "dec"
    with_in = month in LAST_QUARTER
    print(f"{month}   or: {str(with_or):<6} in: {str(with_in):<6} {'ok' if with_or == with_in else 'DIFFER'}")
```

Both always agree. The `in` version wins when the policy changes: if the quarter now starts in
September, one element gets added to the list and no condition is touched.

### Exercise 5

```python
TITLES = ["Sales analyst", "Brand Manager", "people manager",
          "Financial Analyst", "Recruiter", "Operations Manager"]

print("Management:")
for t in TITLES:
    if "manager" in t.lower():
        print("  ", t)

print("Analysts:")
for t in TITLES:
    if "analyst" in t.lower():
        print("  ", t)
```

The `.lower()` goes on the title, not on the word being searched for. The order is what matters:
you normalise the data and compare against a value you already wrote in lowercase.

### Exercise 6

```python
one = {"channel": "Instagram", "clicks": 5074}
two = {"channel": "Instagram", "clicks": 5074}

print("one == two :", one == two)
print("one is two :", one is two)

three = one
print("three is one:", three is one)

three["clicks"] = 9999
print("one after touching three:", one)

# is would be the right question when what you want to know is whether two names
# point at the same object, because then modifying one modifies the other. The
# last line proves it: touching three changed one, and that only happens when is
# comes back true.
```

That behaviour is the underlying reason `.copy()` exists in pandas, and why week 15.2 starts by
calling `sales.copy()` before the demonstration.

### Exercise 7

```python
def can_ship(weight, destination, paid):
    return "ship" if (paid and weight <= 20 and destination != "international") else "do not ship"


CASES = [(15, "national", True), (15, "national", False),
         (25, "national", True), (25, "national", False),
         (15, "international", True), (15, "international", False),
         (25, "international", True), (20, "local", True)]

for weight, destination, paid in CASES:
    print(f"{weight:>3} kg {destination:<15} {'paid' if paid else 'unpaid':<8} -> {can_ship(weight, destination, paid)}")
```

Nine lines and three levels of indentation became one. The signal was in plain sight: all three
`else` clauses returned exactly the same thing.

### Exercise 8

There is no published solution, because the policy is different for everyone. It is graded on four
things: that all three operators are used, that indentation does not go past two levels, that all
four rows of the truth table are tested, and that the condition can be read out loud without
ambiguity.